# GraphRAG: a hands-on implementation

**Track:** Search · **Guide:** [`049-graphrag.md`](../guides/concepts/049-graphrag.md)

GraphRAG uses entity relationships and graph structure alongside text retrieval to answer questions spanning connected facts.

> **Intuition:** Instead of searching only for paragraphs, follow a map of people, events, and their relationships.

This lesson builds a small working example in stages: prepare inputs, implement
the core idea, run it on a realistic toy case, and inspect its output and limits.
It runs offline unless a cell is explicitly marked as an optional provider call.

```mermaid
flowchart LR
    A[Prepare inputs] --> B[Implement GraphRAG]
    B --> C[Run on example]
    C --> D[Evaluate and inspect]
```

**Case study:** Summarize a project history by finding linked teams, decisions, and incidents across many documents.

Run the cells from top to bottom. To install the notebook environment, follow
[`guides/SETUP.md`](../guides/SETUP.md). The toy data and implementation are for
learning, not a production quality, safety, or security guarantee.


## 1. Create source-linked entity facts

GraphRAG represents entities and relationships extracted from source documents, retaining provenance for each edge.


In [ ]:
edges = [
    {"source": "Team-R", "relation": "owns", "target": "Project-P", "doc": "plan-1"},
    {"source": "Project-P", "relation": "decided", "target": "Decision-7", "doc": "minutes-4"},
    {"source": "Decision-7", "relation": "caused", "target": "Incident-2", "doc": "review-2"},
    {"source": "Team-S", "relation": "supports", "target": "Project-Q", "doc": "plan-2"},
]

## 2. Retrieve a relevant subgraph

Start from a query-linked entity, expand a bounded number of hops, and carry source IDs along with facts.


In [ ]:
def expand(seed, max_hops=2):
    frontier = {seed}
    seen = set(frontier)
    selected = []
    for _ in range(max_hops):
        next_frontier = set()
        for edge in edges:
            if edge["source"] in frontier:
                selected.append(edge)
                next_frontier.add(edge["target"])
        frontier = next_frontier - seen
        seen.update(frontier)
    return selected

subgraph = expand("Team-R", max_hops=3)
print("retrieved edges:", subgraph)

## 3. Summarize with evidence

A grounded response reports connected facts and source IDs. Graph completeness and extraction quality determine whether an answer can be trusted.


In [ ]:
summary = [
    f"{edge['source']} {edge['relation']} {edge['target']} [source: {edge['doc']}]"
    for edge in subgraph
]
print("\n".join(summary))
assert any(edge["target"] == "Incident-2" for edge in subgraph)

## Wrap-up and next steps

- Identify what was learned, stored, retrieved, or generated in this implementation.
- Change an input and observe how the output changes.
- Test one failure case and decide when the system should abstain or ask for help.
- Replace toy data/components only after you can evaluate the change against a
  held-out or representative test set.

Continue with the [concept guide](../guides/concepts/049-graphrag.md) and the
[learning roadmap](../guides/ROADMAP.md).
